# Diabetes Risk Screening using an Artificial Neural Network

This notebook is the reproducible analysis companion to the GitHub project. It uses the same modules and artifacts as the Streamlit app.

> **Healthcare disclaimer:** Educational portfolio demonstration only. Not a diagnostic tool or medical advice.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR, MODEL_DIR, OUTPUT_DIR
from src.data_preprocessing import load_dataset, create_data_quality_report
from src.prediction_pipeline import DiabetesPredictionPipeline

2026-07-15 17:36:23.762594: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-15 17:36:23.920922: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1. Load and audit the dataset

In [2]:
data = load_dataset(DATA_DIR / "diabetes.csv")
data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
quality_report = create_data_quality_report(data)
pd.Series(quality_report["zero_as_missing_counts"], name="Zero markers")

Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
Name: Zero markers, dtype: int64

Zeros in glucose, blood pressure, skin thickness, insulin, and BMI are converted to missing values. Median imputation and scaling are fitted on training data only. Zero pregnancies remains valid.

## 2. Reproduce training (optional)

In [4]:
# Training is deterministic but may take a few minutes on CPU.
# Uncomment to regenerate models/ and outputs/.
# from src.model_training import train
# metadata = train(DATA_DIR / "diabetes.csv")

## 3. Review saved model metadata and held-out metrics

In [5]:
metadata = json.loads((MODEL_DIR / "model_metadata.json").read_text())
metrics = pd.Series(metadata["test_metrics"])
metrics[["classification_threshold", "accuracy", "precision", "recall", "f1_score", "roc_auc", "pr_auc"]]

classification_threshold        0.33
accuracy                    0.741379
precision                    0.59322
recall                      0.853659
f1_score                         0.7
roc_auc                      0.85561
pr_auc                      0.772281
dtype: object

The classification threshold is selected on the validation set using F2, giving recall more weight than precision for this screening demonstration. Low/Medium/High bands are separate communication ranges.

## 4. Inspect permutation importance

In [6]:
importance = pd.read_csv(OUTPUT_DIR / "feature_importance.csv")
importance

,feature,importance_mean_auc_decrease,importance_std
0,Glucose,0.175886,0.042709
1,BMI,0.061789,0.012992
2,Age,0.018976,0.008050
3,Pregnancies,0.008715,0.014098
4,Insulin,0.007496,0.009584
5,BloodPressure,0.004179,0.008538
6,SkinThickness,0.002699,0.008366
7,DiabetesPedigreeFunction,0.000390,0.012548


Permutation importance describes model dependence in this held-out sample and must not be interpreted as clinical causality.

## 5. Score example records with the deployment pipeline

In [7]:
pipeline = DiabetesPredictionPipeline()
sample = pd.read_csv(DATA_DIR / "sample_input.csv")
scored = pipeline.score(sample)
scored

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,DiabetesRiskProbability,RiskCategory,ScreeningFlag,Interpretation
0,1,85,66,29,0,26.6,0.351,31,0.042775,Low Risk,No elevated screening flag,The model estimates a lower relative risk for ...
1,6,148,72,35,0,33.6,0.627,50,0.822105,High Risk,Elevated screening flag,The model estimates a higher screening risk. C...
2,0,167,0,0,0,32.3,0.839,30,0.923236,High Risk,Elevated screening flag,The model estimates a higher screening risk. C...
3,4,110,76,20,100,28.4,0.118,27,0.085686,Low Risk,No elevated screening flag,The model estimates a lower relative risk for ...
4,9,171,110,24,240,45.4,0.721,54,0.903714,High Risk,Elevated screening flag,The model estimates a higher screening risk. C...


## Conclusion

The portfolio version improves the original notebook by adding medical zero handling, class weights, recall-focused threshold tuning, a complete preprocessing artifact, PR-AUC, explainability, batch inference, tests, CI, and a hosted Streamlit interface. The project remains an educational demonstration and not a clinical system.